このNotebookは、モデルを学習させるために作られたものである。

# 1. Import

In [47]:
import os
import random
import warnings
warnings.filterwarnings('ignore')
from typing import List, Dict, Optional, Tuple
from IPython.display import display
import datetime
import time
from tqdm.notebook import tqdm

# Data handling
import numpy as np
import polars as pl
import pandas as pd
from sklearn.model_selection import  StratifiedKFold

# Medical imaging
import cv2

# Machine Lerning 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast
import timm

# Transformations
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Visualization
import matplotlib.pyplot as plt

# Experiment Management
import wandb

# Competition API
# import kaggle_evaluation.rsna_inference_server

# 2. Configuration

In [48]:
# datetime for unique checkpoint filenames
date_time = datetime.datetime.now()
date_time = date_time.strftime('%Y-%m-%d_%H-%M-%S')

In [49]:
# Set device
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
    torch.cuda.empty_cache()
    DEVICE = 'cuda'
else:
    raise RuntimeError("CUDA is not available! This code requires GPU.")

GPU: NVIDIA GeForce RTX 4090
Memory: 24.0 GB
CUDA version: 12.4


In [50]:
class Configuration:
    
    # Run
    run_name = "swin-s-5folds"
    save_dir = "../outputs"
    test_run = False
    seed = 42
    device = DEVICE
    
    # Resumption
    checkpoint_dir = ""
    checkpoint_fold = -1 # 0～(num_folds-1)
    checkpoint_epoch = -1 # 0～(num_epochs-1)    
    
    # Model
    pretrained = True
    
    # Input Data
    image_size = 512
    num_slices = 32
    use_aggregated_slices = False
    batch_size = 5
    num_folds = 5
    label_names = [
        'Left Infraclinoid Internal Carotid Artery',
        'Right Infraclinoid Internal Carotid Artery',
        'Left Supraclinoid Internal Carotid Artery',
        'Right Supraclinoid Internal Carotid Artery',
        'Left Middle Cerebral Artery',
        'Right Middle Cerebral Artery',
        'Anterior Communicating Artery',
        'Left Anterior Cerebral Artery',
        'Right Anterior Cerebral Artery',
        'Left Posterior Communicating Artery',
        'Right Posterior Communicating Artery',
        'Basilar Tip',
        'Other Posterior Circulation',
        'Aneurysm Present',
        ]
    num_labels = len(label_names)
    
    # Training
    num_epochs = 10
    patience = 2
    pos_weight = torch.tensor(
        [
        54.743589743589745,
        43.36734693877551,
        12.13595166163142,
        14.696750902527075,
        18.85388127853881,
        13.789115646258503,
        10.977961432506888,
        93.52173913043478,
        76.64285714285714,
        49.55813953488372,
        42.04950495049505,
        38.527272727272724,
        37.47787610619469,
        1.332618025751073
        ]).to(device, dtype=torch.float32)
    
    # Rename run_name to add further details
    run_name = run_name + f'-{image_size}-{num_slices}'
    save_dir = save_dir + '/' + run_name + f'-{date_time}'
    
    if checkpoint_dir != "":
        run_name = checkpoint_dir
        save_dir = checkpoint_dir
    
    # Weights & Biases
    if test_run:
        use_wandb = False
        wandb_init = {}
        artifact = {}
    else:
        use_wandb = True
        wandb_init = {
            'project': 'RSNA-IAD',
            'group': 'Image Classification',
            'job_type': 'training model',
            'save_code': True,
        }
        artifact = {
            'name': run_name + date_time,
            'type': 'model, optimizer, scheduler',
        }

CFG = Configuration

In [51]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
    torch.cuda.empty_cache()
    CFG.device = 'cuda'
    CFG.pos_weight = CFG.pos_weight.to(CFG.device, dtype=torch.float32)
else:
    raise RuntimeError("CUDA is not available! This code requires GPU.")

Using device: cuda
GPU: NVIDIA GeForce RTX 4090
Memory: 24.0 GB
CUDA version: 12.4


In [52]:
def set_random_seed(seed=CFG.seed, deterministic=True):
    """
    Set random seed.
    
    Args:
        seed (int): Seed to be used.
        deterministic (bool): Whether to set the deterministic option for
            CUDNN backend, i.e., set `torch.backends.cudnn.deterministic`
            to True and `torch.backends.cudnn.benchmark` to False.
            Default: False.
    """
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    
    if deterministic:
        torch.backends.cudnn.benchmark = True


In [53]:
set_random_seed(seed=CFG.seed, deterministic=True)


# 3. Weights & Biases

In [54]:
if CFG.use_wandb:
    os.environ['WANDB_NOTEBOOK_NAME'] = CFG.run_name
    wandb.login()
    run = wandb.init(**CFG.wandb_init)
    artifact = wandb.Artifact(**CFG.artifact)
else:
    run = None
    artifact = None


In [55]:
def alert_by_wandb(title='', text=''):
    wandb.alert(title, text)


# 4. Model

In [56]:
class SwinWithMetaModel(nn.Module):
    
    def __init__(self, model_name, pretrained=CFG.pretrained,
                 num_classes=CFG.num_labels,
                 drop_rate=0.3, drop_path_rate=0.2):
        super().__init__()
        self.model_name = model_name
        
        if model_name == 'swin_s':
            self.backbone = timm.create_model(
                'swin_small_patch4_window7_224',
                pretrained=pretrained,
                img_size=CFG.image_size,
                num_classes=0,
                drop_rate=drop_rate,
                drop_path_rate=drop_path_rate,
                global_poopling='')
            
            # input layer modification: 3 channels -> CFG.num_slices channels
            self.backbone.patch_embed.proj = nn.Conv2d(
                in_channels=CFG.num_slices,
                out_channels=96,
                kernel_size=4,
                stride=4,
            )
        else:
            raise ValueError(f"Model {model_name} is not supported.")
        
        # with torch.no_grad():
        #     dummy_input = torch.zeros(
        #         1,
        #         CFG.num_slices,
        #         CFG.image_size,
        #         CFG.image_size
        #         )
        #     features = self.backbone(dummy_input)
            
        #     if len(features.shape) == 4:
        #         # Conv features (batch, channels, height, width)
        #         num_features = features.shape[1]
        #         self.needs_pool = True
        #     elif len(features.shape) == 3:
        #         # Transformer features (batch, sequence, features)
        #         num_features = features.shape[-1]
        #         self.needs_pool = False
        #         self.needs_seq_pool = True
        #     else:
        #         # Already flat features (batch, features)
        #         num_features = features.shape[1]
        #         self.needs_pool = False
        #         self.needs_seq_pool = False
        # print(f"Model name: {model_name}")
        # print(f"Features: {num_features}")
        # print(f"Output Shape: {features.shape}")
        
        # # Add global pooling for models that output spatial features
        # if self.needs_pool:
        #     self.global_pool = nn.AdaptiveAvgPool2d(1)
        
        self.meta_features = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 32),
            nn.ReLU()
        )
        
        # According to "LB #1"
        self.classifier = nn.Sequential(
            nn.Linear(768 + 32, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, images, meta):
        # Extract image features
        image_features = self.backbone(images)
        
        # # Apply appropriate pooling based on model type
        # if hasattr(self, 'needs_pool') and self.needs_pool:
        #     # Conv features - apply global pooling
        #     image_features = self.global_pool(image_features)
        #     image_features = image_features.flatten(1)
        # elif hasattr(self, 'needs_seq_pool') and self.needs_seq_pool:
        #     # Transformer features - average across sequence dimension
        #     image_features = image_features.mean(dim=1)
        # elif len(image_features.shape) == 4:
        #     # Fallback for any 4D output
        #     image_features = F.adaptive_avg_pool2d(image_features, 1).flatten(1)
        # elif len(image_features.shape) == 3:
        #     # Fallback for any 3D output
        #     image_features = image_features.mean(dim=1)
        
        # Process Meta Features
        meta_fieatures = self.meta_features(meta)
        
        # Combine Features
        x = torch.cat([image_features, meta_fieatures], dim=1)
        
        # Classicication
        x = self.classifier(x)
        x = torch.nn.Sigmoid()(x)
        return x

model = SwinWithMetaModel(model_name='swin_s', pretrained=CFG.pretrained)

-> timm.createmodel(num_classes=0)とすると、最後のnn.Linear()がnn.Identity()になる。

In [57]:
model.to(CFG.device)

is_in_cuda_list = []

for name, parameter in model.named_parameters():
    # determination of cuda and its storage
    is_in_cuda_list.append(parameter.is_cuda)
    
if all(is_in_cuda_list):
    print('All parameters is in cuda')
        
else:
    print('One of the parameters is not in the cuda.')


All parameters is in cuda


# 5. Criterion

In [58]:
class FocalLoss(nn.Module):
    """Focal Loss for addressing class imbalance"""
    def __init__(self, alpha=1, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        
    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * bce_loss
        return focal_loss.mean()

class WeightedMultiLabelLoss(nn.Module):
    """Weighted multi-label loss"""
    def __init__(self, aneurysm_weight=3.0):
        super(WeightedMultiLabelLoss, self).__init__()
        self.weights = torch.ones(CFG.num_labels, device=device)
        # self.weights[-1] = aneurysm_weight
        
    def forward(self, outputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(outputs, targets, reduction='none')
        weighted_loss = bce_loss * self.weights
        return weighted_loss.mean()

In [59]:
class ImprovedLoss(nn.Module):
    """Advanced combined loss function"""
    def __init__(self, aneurysm_weight=3.0, focal_weight=0.3):
        super(ImprovedLoss, self).__init__()
        self.aneurysm_weight = aneurysm_weight
        self.focal_weight = focal_weight
        
        # self.weights = torch.ones(CFG.num_labels, device=device) # ← Original
        self.weights = CFG.pos_weight # ← modified
        
        # self.weights[-1] = aneurysm_weight
        
        self.focal_loss = FocalLoss(alpha=1, gamma=2)
        
    def forward(self, outputs, targets):
        # Weighted BCE
        bce_loss = F.binary_cross_entropy_with_logits(outputs,
                                                      targets,
                                                      reduction='none')
        weighted_bce = (bce_loss * self.weights).mean()
        
        # Focal Loss
        focal_loss = self.focal_loss(outputs, targets)
        
        # Combination
        loss = (1 - self.focal_weight) * weighted_bce \
            + self.focal_weight * focal_loss
            
        return loss
            

In [60]:
def build_models():
    
    # Model
    model = SwinWithMetaModel(model_name='swin_s', pretrained=CFG.pretrained)
    model.to(CFG.device)
    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters())
    # Loss Function
    criterion = nn.BCEWithLogitsLoss(pos_weight=CFG.pos_weight)
    # criterion = ImprovedLoss(aneurysm_weight=3.0, focal_weight=0.3)
    # Schedulers
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=CFG.num_epochs,
        eta_min=1e-6
    )
    
    return model, optimizer, criterion, scheduler

# 6. Dataset

In [61]:
# SeriesInstanceUID list
series_list = os.listdir(f'../series_npy/{CFG.image_size}')

# .npy path DataFrame
image_path_df = pd.read_csv(
    f'../npy_path/image_{CFG.image_size}_path_df.csv'
)

# Meta DataFrame
meta_df = pd.read_csv('../meta_data/meta.csv')

# Label DataFrame
label_df = pd.read_csv(f'../train.csv')
label_df = label_df[['SeriesInstanceUID'] + CFG.label_names]


In [62]:
# for training
train_transform = A.Compose(
    [   
        # Rotation
        A.Rotate(limit=(-3, 3), p=0.5, border_mode=cv2.BORDER_WRAP,  # cv2.BORDER_WRAP,
                 seed=CFG.seed
        ),
        
        # Normalization
        A.Normalize(normalization='min_max'),
        
        # ToTensor
        ToTensorV2(),
    ]
)

# for inference
inference_transform = A.Compose(
    [
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ]
)
    
# for TTA
tta_transform = A.Compose(
    [
        # Rotation
        A.Rotate(limit=(-3, 3), p=1.0, border_mode=cv2.BORDER_WRAP, 
                 seed=CFG.seed
                 ),
        
        # Normalization
        A.Normalize(normalization='min_max'),
        
        # ToTensor
        ToTensorV2(),
        
        # A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            
        # # Horizontal flip
        # A.HorizontalFlip(p=1.0),
        # A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        
        # # Vertical flip
        # A.VerticalFlip(p=1.0),
        # A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        
        # # 90 degree rotation
        # A.RandomRotate90(p=1.0),
        # A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        
        # # ↓ Original
        # # Sharpen
        # A.Sharpen(alpha=(0, 1.0), p=1.0),
        # A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        
        # ToTensorV2(),
    ]
)


In [63]:
class RSNAIADDataset(torch.utils.data.Dataset):
    '''
    Datasetの__getitem__()は、num_slicesの枚数分だけ画像を出力する。
    
    Arguments:
    - series_list: 画像のSeriesInstanceUIDのリスト
    - image_path_df: 画像のパスを含むDataFrame
    - meta_df: 患者のメタデータが入ったDataFrame
    - label_df: ラベルが入ったDataFrame
    - num_slices: 1つのシリーズから抽出するスライス数
    - transforms: 画像変換のためのAlbumentationsのComposeオブジェクト
    '''
    def __init__(self,
                 series_list: list,
                 image_path_df=image_path_df,
                 meta_df=meta_df,
                 label_df=label_df,
                 transforms=None
        ):
        self.series_list = series_list
        self.image_path_df = image_path_df
        self.meta_df = meta_df
        self.label_df = label_df
        self.transforms = transforms
        self.num_slices = CFG.num_slices
        self.use_aggregated_slices = CFG.use_aggregated_slices

    def __len__(self):
        return len(self.series_list)

    def __getitem__(self, index):
        # Index to SeriesInstanceUID
        series_id = self.series_list[index]
        
        # Extract image paths from DataFrame
        image_path_df = self.image_path_df.loc[
            self.image_path_df['series_id'] == series_id
        ].reset_index(drop=True)
        
        # Load Images
        indices = np.linspace(0,
                              len(image_path_df) - 1,
                              self.num_slices).astype(np.int32)
        # Stack images to (H, W, CFG.num_slices)
        images = []
        for i in indices:
            image_path = image_path_df.loc[i, 'npy_path']
            image = np.load(image_path).astype(np.uint8)
            images.append(image)
        images = np.stack(images, axis=-1)
        
        # Transform
        if self.transforms:
            # ToTensorV2はnumpy.ndarrayをtorch.Tensorに変換する
            augmented = self.transforms(image=images)
            images = augmented['image']
        else:
            images = torch.tensor(images, dtype=torch.float32)
            images = torch.permute(images, (2, 0, 1))
            # Min-Max Normalization
            if torch.max(images) > 1.0:
                max_value = torch.max(images)
                min_value = torch.min(images)
                images = (images - min_value) / (max_value - min_value)
                
        # Meta data
        meta = self.meta_df.loc[
            self.meta_df['SeriesInstanceUID'] == series_id, ['age', 'sex']
        ]
        age = min(meta['age'].values[0], 100)
        age = age / 100
        sex = meta['sex'].values[0]
        meta = torch.tensor([age, sex], dtype=torch.float32)

        # Labels
        labels = self.label_df.loc[
            self.label_df['SeriesInstanceUID']==series_id, \
                CFG.label_names].values
        labels = torch.tensor(labels, dtype=torch.float32)
        labels = torch.squeeze(labels, dim=0)
        
        return images, meta, labels, series_id


# 7. DataLoader

In [64]:
# Convert multi-label data into a single numerical ID
label_df['label_id'] = 0
digits = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192]
digits = digits[:CFG.num_labels]
for i, label_name in enumerate(CFG.label_names):
    label_df['label_id'] += label_df[label_name] * digits[i]
    
# Stratified K-Fold
skf = StratifiedKFold(
    n_splits=CFG.num_folds,
    shuffle=True,
    random_state=CFG.seed
)
label_df['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(\
    skf.split(X=label_df, y=label_df['label_id'])):
    label_df.loc[val_idx, 'fold'] = fold

In [65]:
def build_dataloaders(fold: int):

    train_series = label_df.loc[\
        label_df['fold']!=fold, "SeriesInstanceUID"].values
    val_series = label_df.loc[\
        label_df['fold']==fold, "SeriesInstanceUID"].values
    train_labels = label_df.loc[label_df['fold']!=fold, CFG.label_names].values
    val_labels = label_df.loc[label_df['fold']==fold, CFG.label_names].values

    if CFG.test_run:
        train_series = train_series[:5]
        val_series = val_series[:5]
        train_labels = train_labels[:5]
        val_labels = val_labels[:5]

    # 2 dimensions -> 1 dimension
    train_series, val_series = train_series.flatten(), val_series.flatten()
    print(f"Train size: {len(train_series)}, Val size: {len(val_series)}")

    # Datasets
    train_dataset = RSNAIADDataset(
        series_list=train_series,
        transforms=train_transform
    )
    val_dataset = RSNAIADDataset(
        series_list=val_series,
        transforms=train_transform # or tta_transform
    )
    
    # DataLoaders
    train_dataloader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=CFG.batch_size,
        shuffle=True,
        num_workers=0
    )
    val_dataloader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=CFG.batch_size,
        shuffle=True,
        num_workers=0
    )

    return train_dataloader, val_dataloader

In [66]:
train_dataloaders = []
val_dataloaders = []

for fold in range(CFG.num_folds):
    print(f"Fold {fold}")
    train_dataloader, val_dataloader = build_dataloaders(fold)
    train_dataloaders.append(train_dataloader)
    val_dataloaders.append(val_dataloader)

Fold 0
Train size: 3478, Val size: 870
Fold 1
Train size: 3478, Val size: 870
Fold 2
Train size: 3478, Val size: 870
Fold 3
Train size: 3479, Val size: 869
Fold 4
Train size: 3479, Val size: 869


In [67]:
# _, ax = plt.subplots(1, 2, figsize=(12, 6))

# # 元の画像とどのくらい違いがあるかを確認

# # 元の画像(.npy)
# src = np.load(f'../series_npy/{CFG.image_size}/1.2.826.0.1.3680043.8.498.10034081836061566510187499603024895557/00012.npy')
# print(np.unique(src))
# ax[0].imshow(src)

# # Datasetから取り出した画像
# images, _ = train_dataset[0]
# image = images[8].numpy()  # shape: [H, W]

# # 0-1のfloatなら0-255に変換
# if image.max() <= 1.0:
#     image = (image * 255).astype(np.uint8)
# else:
#     image = image.astype(np.uint8)

# ax[1].imshow(image)


In [68]:
# pil_image = Image.fromarray(image)
# display(pil_image)


# 8. Functions

In [69]:
# count execution time for one epoch
def count_time(start:float) -> float:
    
    elapsed_time = time.time() - start
    elapsed_time /= 60
    
    return elapsed_time


In [70]:
# to save model, optimizer, scheduler
def save_checkpoint(model, optimizer, scheduler,
                    fold=100, save_dir=CFG.save_dir):
    
    model.to('cpu')
    
    model_state_dict =  model.state_dict()
    optimizer_state_dict = optimizer.state_dict()
    scheduler_state_dict = scheduler.state_dict()
    
    model_path = save_dir + f'/model_fold{fold}.pth'
    optimizer_path = save_dir + f'/optimizer_fold{fold}.pth'
    scheduler_path = save_dir + f'/scheduler_fold{fold}.pth'
        
    torch.save(model_state_dict, model_path)
    torch.save(optimizer_state_dict, optimizer_path)
    torch.save(scheduler_state_dict, scheduler_path)
    
    model.to(device)
    
    print(f"Model saved.")

# to load model, optimizer, scheduler
def load_checkpoint(model, optimizer, scheduler,
                    checkpoint_dir=CFG.checkpoint_dir, fold=100):
    
    model.to('cpu')
    
    model.load_state_dict(checkpoint_dir + f'/model{fold}.pth')
    optimizer.load_state_dict(checkpoint_dir + f'/optimizer{fold}.pth')
    scheduler.load_state_dict(checkpoint_dir + f'/scheduler{fold}.pth')
    
    model.to(device)
    
    return model, optimizer, scheduler

In [71]:
def add_files_to_artifact(fold=100, save_dir=CFG.save_dir):
    
    artifact.add_file(save_dir + f'/model_fold{fold}.pth')
    artifact.add_file(save_dir + f'/optimizer_fold{fold}.pth')
    artifact.add_file(save_dir + f'/scheduler_fold{fold}.pth')
    
    print("Files added to the artifact.")

# 9. Training

In [72]:
def train_one_epoch(model, optimizer, scheduler, criterion,
                    train_dataloader, val_dataloader,
                    epoch=100) -> Tuple[float, float, List, List]:
    
    print(f'---------- Epoch {epoch} ----------')
    
    # Training
    model.train()
    train_losses = []
    
    for images, meta, labels, series_ids in tqdm(train_dataloader):
        images = images.to(CFG.device)
        meta = meta.to(CFG.device)
        labels = labels.to(CFG.device)
        optimizer.zero_grad()
        with autocast(device_type=CFG.device):
            outputs = model(images, meta)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
    
    mean_train_loss = np.mean(train_losses)
    print(f'Inner Mean Train Loss: {mean_train_loss:.4f}')
    
    # Validation
    model.eval()
    val_losses = []
    inner_series_ids_list = []
    inner_predicted_list = []
    
    with torch.no_grad():
        for images, meta, labels, series_ids in tqdm(val_dataloader):
            images = images.to(CFG.device)
            meta = meta.to(CFG.device)
            labels = labels.to(CFG.device)
            with autocast(device_type=CFG.device):
                outputs = model(images, meta)
                loss = criterion(outputs, labels)
                val_losses.append(loss.item())
                inner_series_ids_list.extend(series_ids)
                inner_predicted_list.extend(outputs.cpu().numpy().tolist())
        
    mean_val_loss = np.mean(val_losses)
    print(f'Inner Mean Validation Loss: {mean_val_loss:.8f}')
    
    scheduler.step()
        
    return mean_train_loss, mean_val_loss,\
        inner_series_ids_list, inner_predicted_list

In [73]:
def main():
    
    if not CFG.test_run:
        os.makedirs(CFG.save_dir, exist_ok=True)
    
    fold_val_losses = []
    outer_series_ids_list = []
    outer_predicted_list = []
    
    for fold in range(CFG.num_folds):
        print(f'======================== Fold {fold} ========================')
        
        # Model, Optimizer, Criterion, Scheduler
        model, optimizer, criterion, scheduler = build_models()
        
        # Dataloaders
        train_dataloader = train_dataloaders[fold]
        val_dataloader = val_dataloaders[fold]
        
        best_predicted_list = []
        best_val_loss = np.inf
        now_patience = 0
    
        for epoch in range(CFG.num_epochs):
            
            # Resume from checkpoint
            if (fold == CFG.checkpoint_fold) and \
                (epoch == CFG.checkpoint_epoch):
                model, optimizer, scheduler = load_checkpoint(
                    model, optimizer, scheduler,
                    directory=CFG.directory_to_resume
                )
                print(f'Resumed from Fold {fold} checkpoint.')
                
            elif fold < CFG.checkpoint_fold:
                print(f'Fold {fold} Epoch {epoch} was skipped.')
                continue
            
            elif (fold == CFG.checkpoint_fold) and \
                (epoch < CFG.checkpoint_epoch):
                print(f'Fold {fold} Epoch {epoch} was skipped.')
                continue
            
            # Train & Validation
            start_time = time.time()
            train_loss, val_loss, inner_series_ids, inner_predicted_list \
                = train_one_epoch(model, optimizer, scheduler, criterion,
                                  train_dataloader, val_dataloader,
                                  epoch=epoch)
            elapsed_time = count_time(start_time)
            print(f'Elapsed time: {elapsed_time}')
            
            # Log Losses to W&B
            if CFG.use_wandb:
                losses = {
                    f'train_loss_fold{fold}': train_loss,
                    f'val_loss_fold{fold}': val_loss
                }
                wandb.log(losses)
            
            # Test Run
            if CFG.test_run:
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_predicted_list = inner_predicted_list
                print('Test run: Skip saving checkpoint.')
            # Not Test Run
            else:
                # Save best checkpoint
                if val_loss < best_val_loss:
                    now_patience = 0
                    best_val_loss = val_loss
                    save_checkpoint(model, optimizer, scheduler, fold=fold)
                    print(f'Best checkpoint saved at {CFG.save_dir}')
                    best_predicted_list = inner_predicted_list
                else:
                    now_patience += 1
                    print(f'Patience: {now_patience}/{CFG.patience}')
                    if now_patience >= CFG.patience:
                        print('Early stopping.')
                        alert_by_wandb(
                            title='Early Stopping',
                            text=f'Fold {fold} stopped early at epoch {epoch}.'
                        )
                        break
        
        # Add model, optimizer, scheduler files to W&B
        if CFG.use_wandb:
            add_files_to_artifact(fold=fold)
        
        # Collect results for all folds
        fold_val_losses.append(best_val_loss)
        
        # Collect results for all folds
        outer_series_ids_list.extend(inner_series_ids)
        outer_predicted_list.extend(best_predicted_list)
        
    print(f'====================== All folds completed ======================')
    mean_fold_val_losses = np.mean(fold_val_losses)
    print(f'Each Fold Validation Losses: {fold_val_losses}')
    print(f'Mean Validation Loss: {mean_fold_val_losses:.8f}')
        
    if CFG.use_wandb:
        # Log Mean Validation Loss to W&B
        wandb.log({'mean_fold_val_loss': mean_fold_val_losses})
        
        # Log all files to W&B
        run.log_artifact(artifact)
        print('All artifacts were logged to W&B')
            
    return outer_series_ids_list, outer_predicted_list

In [74]:
outer_series_ids_list, outer_predicted_list = main()

======================== Fold 0 ========================
---------- Epoch 0 ----------


  0%|          | 0/696 [00:00<?, ?it/s]

Inner Mean Train Loss: 1.3034


  0%|          | 0/174 [00:00<?, ?it/s]

Inner Mean Validation Loss: 1.28204660
Elapsed time: 43.79202872514725
Model saved.
Best checkpoint saved at ../outputs/swin-s-5folds-512-32-2025-10-21_16-44-44
---------- Epoch 1 ----------


...

Inner Mean Validation Loss: 1.29996627
Elapsed time: 33.079448727766675
Patience: 1/2
---------- Epoch 3 ----------


  0%|          | 0/696 [00:00<?, ?it/s]

Inner Mean Train Loss: 1.2941


  0%|          | 0/174 [00:00<?, ?it/s]

Inner Mean Validation Loss: 1.29701213
Elapsed time: 32.320479861895244
Patience: 2/2
Early stopping.
Files added to the artifact.
====================== All folds completed ======================
Each Fold Validation Losses: [1.279914174956837, 1.2873001095207257, 1.3012378900900654, 1.3027198122835708, 1.296517560536834]
Mean Validation Loss: 1.29353791
All artifacts were logged to W&B


In [75]:
outer_predicted_list[:3]

[[0.0221099853515625,
  0.01560211181640625,
  0.017181396484375,
  0.043609619140625,
  0.00626373291015625,
  0.014556884765625,
  0.013427734375,
  0.01056671142578125,
  0.00128936767578125,
  0.0163421630859375,
  0.006931304931640625,
  0.011871337890625,
  0.00714874267578125,
  0.042572021484375],
 [0.015960693359375,
  0.0120086669921875,
  0.0120086669921875,
  0.03533935546875,
  0.004486083984375,
  0.011199951171875,
  0.01090240478515625,
  0.00847625732421875,
  0.0008625984191894531,
  0.01332855224609375,
  0.005344390869140625,
  0.00812530517578125,
  0.00536346435546875,
  0.0328369140625],
 [0.0254669189453125,
  0.017578125,
  0.0203399658203125,
  0.047943115234375,
  0.00731658935546875,
  0.0165863037109375,
  0.01495361328125,
  0.0115966796875,
  0.0015611648559570312,
  0.0180206298828125,
  0.00787353515625,
  0.01433563232421875,
  0.00821685791015625,
  0.048126220703125]]

# 10. Save Predictions

In [76]:
predicted_df = pd.DataFrame()

predicted_df['SeriesInstanceUID'] = outer_series_ids_list
predicted_df[CFG.label_names] = outer_predicted_list

if not CFG.test_run:
    if CFG.checkpoint_dir == "":
        predicted_df.to_csv(f'{CFG.save_dir}/predicted_labels.csv', index=False)
    else:
        past_df = pd.read_csv(f'{CFG.checkpoint_dir}/predicted_labels.csv')
        concat_df = pd.concat([past_df, predicted_df], axis=0)
        concat_df.to_csv(f'{CFG.save_dir}/predicted_labels.csv', index=False)
        predicted_df = concat_df.copy()

In [77]:
predicted_df.describe()

,Left Infraclinoid Internal Carotid Artery,Right Infraclinoid Internal Carotid Artery,Left Supraclinoid Internal Carotid Artery,Right Supraclinoid Internal Carotid Artery,Left Middle Cerebral Artery,Right Middle Cerebral Artery,Anterior Communicating Artery,Left Anterior Cerebral Artery,Right Anterior Cerebral Artery,Left Posterior Communicating Artery,Right Posterior Communicating Artery,Basilar Tip,Other Posterior Circulation,Aneurysm Present
count,4348.000000,4348.000000,4348.000000,4348.000000,4348.000000,4348.000000,4348.000000,4348.000000,4348.000000,4348.000000,4348.000000,4348.000000,4348.000000,4348.000000
mean,0.023487,0.021877,0.048166,0.040862,0.019153,0.028823,0.024442,0.018991,0.013622,0.024913,0.040906,0.016477,0.020067,0.040518
std,0.015457,0.014260,0.047072,0.035311,0.013613,0.020314,0.014546,0.011774,0.014085,0.020119,0.047089,0.015907,0.014603,0.036241
min,0.000136,0.000368,0.000251,0.001081,0.000152,0.000813,0.001942,0.000285,0.000034,0.000641,0.000342,0.000175,0.000347,0.001949
25%,0.013794,0.013687,0.019302,0.015671,0.007347,0.016586,0.014954,0.011642,0.003197,0.008255,0.007488,0.007935,0.008217,0.010986
50%,0.025085,0.019531,0.023697,0.026764,0.014061,0.024185,0.022415,0.015396,0.007233,0.022537,0.012772,0.010094,0.016663,0.025803
75%,0.033966,0.039276,0.104401,0.063477,0.027954,0.050812,0.036835,0.029709,0.030960,0.035278,0.066711,0.020691,0.028549,0.063721
max,0.061768,0.049042,0.157959,0.138306,0.058136,0.080933,0.062317,0.044922,0.047424,0.063599,0.133179,0.065369,0.057800,0.138672


# 11. Finish

In [78]:
if CFG.use_wandb:
    run.finish()


mean_fold_val_loss,▁
train_loss_fold0,█▃▃▃▂▃▂▂▂▁
train_loss_fold1,█▃▂▂▂▂▂▂▁▁
train_loss_fold2,█▂▂▂▂▁▁
train_loss_fold3,█▃▂▁
train_loss_fold4,█▂▁▁
val_loss_fold0,██▄▃▂▂▂▁▁▁
val_loss_fold1,█▄▃▃▂▂▁▁▁▁
val_loss_fold2,█▂▂▂▁▂▂
val_loss_fold3,█▁▁▄
val_loss_fold4,▆▁█▂
